In [6]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import plotly.graph_objects as go
from sklearn.manifold import TSNE

In [7]:
MODEL = "gpt-6-luna"
db_name = "vector_db"

# Chunks

Our small knowledge base (about 35,000 tokens) would still fit into the context window. Real knowledge bases don't, and sending everything with every question gets slow and expensive.

Instead, we split every document into small pieces called **chunks**. Later, we only send the few chunks that are relevant to the user's question to the LLM.

## Load the knowledge base

We load the Markdown files from the newest run of notebook 1. Each file becomes a LangChain `Document`: its text (`page_content`) plus some `metadata`.

We add a `doc_type` ("jobs" or "companies") to the metadata, so we can color the vectors by type later.

In [9]:
# Each run of notebook 1 has its own dated folder.
# The newest run sorts last because the folder names start with the date.
runs = sorted(glob.glob("knowledge-base/*"))
latest_run = runs[-1]
folders = glob.glob(f"{latest_run}/*")  # The "jobs" and "companies" subfolders
print(f"Loading knowledge base from {latest_run}")

documents = []  # Collects the loaded documents from all folders
for folder in folders:  # Go through each subfolder
    doc_type = os.path.basename(folder)  # "jobs" or "companies"
    loader = DirectoryLoader(  # Create a loader for this folder
        folder,  # Folder to load files from
        glob="**/*.md",  # Only load Markdown files, including in subfolders
        loader_cls=TextLoader,  # Read each file as plain text
        loader_kwargs={"encoding": "utf-8"},  # Read files as UTF-8
    )
    folder_docs = loader.load()  # Load all files into LangChain Document objects
    for doc in folder_docs:  # Go through each loaded document
        doc.metadata["doc_type"] = doc_type  # Tag the document with its type
        documents.append(doc)  # Add the document to the full list

print(f"Loaded {len(documents)} documents")  # Show how many documents were loaded

Loading knowledge base from knowledge-base/2026-09-24_0926
Loaded 32 documents


In [10]:
documents[0]  # Show the first document to verify it was loaded correctly

Document(metadata={'source': 'knowledge-base/2026-09-24_0926/jobs/18-optum-associate-ai-ml-engineer.md', 'doc_type': 'jobs'}, page_content='---\ntitle: "Associate AI/ML Engineer"\ncompany: "Optum"\ncompany_file: "companies/optum.md"\nlocation: "Eden Prairie, MN"\njob_url: "https://www.linkedin.com/jobs/view/4470969396"\nscraped_at: "2026-09-24_0926"\n---\n\n# Associate AI/ML Engineer\n\nOptum is a global organization that delivers care, aided by technology to help millions of people live healthier lives. The work you do with our team will directly improve health outcomes by connecting people with the care, pharmacy benefits, data and resources they need to feel their best. Here, you will find a culture guided by diversity and inclusion, talented peers, comprehensive benefits and career development opportunities. Come make an impact on the communities we serve as you help us advance health equity on a global scale. Join us to start\n **Caring. Connecting. Growing together.**\n As an\n *

## Split the documents into chunks

`RecursiveCharacterTextSplitter` tries to split at natural boundaries first (paragraphs, then lines, then words), so chunks rarely end in the middle of a sentence.

- `chunk_size=1000`: each chunk has at most about 1,000 **characters** (not tokens).
- `chunk_overlap=200`: neighboring chunks share up to 200 characters, so information at a chunk border isn't lost.

A token is about 4 characters in English text. In our knowledge base, it's closer to 5, so a 1,000-character chunk is about 200 tokens, or roughly a fifth of a job posting.

Chunk size is a tradeoff:

- **Too small** (e.g. one sentence): a chunk like "The salary is 80k" no longer says which job it belongs to.
- **Too large** (e.g. a whole job posting): one vector has to represent the salary, the tech stack, and everything else, so it matches none of them precisely.
- **In between**: "salary" and "tech stack" usually end up in different chunks, but each chunk still makes sense on its own.

Each chunk keeps the metadata of its original document.

In [11]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 257 chunks
First chunk:

page_content='---
title: "Associate AI/ML Engineer"
company: "Optum"
company_file: "companies/optum.md"
location: "Eden Prairie, MN"
job_url: "https://www.linkedin.com/jobs/view/4470969396"
scraped_at: "2026-09-24_0926"
---

# Associate AI/ML Engineer' metadata={'source': 'knowledge-base/2026-09-24_0926/jobs/18-optum-associate-ai-ml-engineer.md', 'doc_type': 'jobs'}


# Vectors

To find the relevant chunks for a question, we need a way to compare texts by **meaning**, not just by matching words.

An **embedding model** turns a text into a **vector**: a long list of numbers. This vector is also called an **embedding**.

The key idea: texts with similar meaning get vectors that are close to each other. "Software engineer role" and "developer position" end up close together, even though they share no words.

Later, we'll embed the user's question the same way and look up the chunks with the closest vectors.

## Create the vector store

We send every chunk to an **embedding model** and store the resulting vectors in **Chroma**, a vector database that runs locally and saves its data to the `vector_db` folder.

We delete the old collection first, so running this cell twice doesn't create duplicates.

The commented-out `HuggingFaceEmbeddings` line is a free alternative that runs on your own machine.

In [12]:
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks, embedding=embeddings, persist_directory=db_name
)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 257 documents


## Inspect the vectors

Each chunk now has exactly one vector. All vectors have the same length (the number of **dimensions**), no matter how long the chunk is. `text-embedding-3-large` produces 3,072 dimensions.

In [13]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 257 vectors with 3,072 dimensions in the vector store


## Visualize the vectors

We can't picture 3,072 dimensions. **t-SNE** squeezes the vectors down to 2 or 3 dimensions while trying to keep similar vectors close together.

First, we load all vectors, chunk texts, and document types from the vector store.

In [14]:
# Prework

result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]
doc_types = [metadata["doc_type"] for metadata in metadatas]

# One color per document type
type_colors = {"jobs": "blue", "companies": "orange"}

Each dot is one chunk. Hover over a dot to see its text.

Notice how job chunks and company chunks form separate groups: the embedding model has captured that they are about different things.

In [15]:

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)
fig = go.Figure()

for doc_type, color in type_colors.items():
    indexes = [i for i, t in enumerate(doc_types) if t == doc_type]
    fig.add_trace(
        go.Scatter(
            x=reduced_vectors[indexes, 0],
            y=reduced_vectors[indexes, 1],
            mode="markers",
            name=doc_type,
            marker=dict(size=5, color=color, opacity=0.8),
            text=[
                f"Type: {doc_type}<br>Text: {documents[i][:100]}..." for i in indexes
            ],
            hoverinfo="text",
        )
    )

fig.update_layout(
    title="2D Chroma Vector Store Visualization",
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40),
)

fig.show()

The same idea in 3D. Drag to rotate the plot.

In [16]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure()

for doc_type, color in type_colors.items():
    indexes = [i for i, t in enumerate(doc_types) if t == doc_type]
    fig.add_trace(
        go.Scatter3d(
            x=reduced_vectors[indexes, 0],
            y=reduced_vectors[indexes, 1],
            z=reduced_vectors[indexes, 2],
            mode="markers",
            name=doc_type,
            marker=dict(size=5, color=color, opacity=0.8),
            text=[
                f"Type: {doc_type}<br>Text: {documents[i][:100]}..." for i in indexes
            ],
            hoverinfo="text",
        )
    )

fig.update_layout(
    title="3D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40),
)

fig.show()